# Pretrain RL Firewall on CIC-IDS2017

Produces `firewall_weights.pth` that drops into `rl_firewall/agent/` and replaces the attack-only weights.

## How to use this notebook on Kaggle

1. Create a new Kaggle notebook (Code > New Notebook).
2. **Add Data** (right panel) → search **`network-intrusion-dataset`** by `chethuhn` — it's the cleanest CIC-IDS2017 mirror with one CSV per attack day. Attach it.
3. **Settings** → Accelerator: **GPU T4 x2** (free). Internet: On.
4. Run all cells. Training takes ~5–15 min depending on subsample size.
5. After the last cell, open the **Output** tab on the right, download `firewall_weights.pth`, and copy it to `rl_firewall/agent/firewall_weights.pth`.

## What it does

- Maps CIC-IDS2017 flow features → the agent's 12-dim state vector (matches `agent/extraction/flow_manager.py`).
- Maps labels → 3-action target: `BENIGN→ALLOW`, slow attacks (`DoS slowloris`, `DoS Slowhttptest`, `Heartbleed`) → `RATE_LIMIT`, everything else → `BLOCK`.
- Trains `FirewallDQN` (same architecture as `agent/dqn/model.py`) by regressing onto Q-targets derived from the same reward table the agent uses at runtime.
- Saves a `state_dict` PyTorch file compatible with `DQNAgent.__init__`'s loader.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load CIC-IDS2017 CSVs

In [ ]:
# Auto-detect the dataset path under /kaggle/input. Adjust DATA_DIR if your
# attached dataset uses a different folder name.
candidates = glob.glob('/kaggle/input/*')
print('Datasets attached:', candidates)

DATA_DIR = next((p for p in candidates if 'intrusion' in p.lower() or 'cicids' in p.lower() or 'cic-ids' in p.lower()), candidates[0])
print('Using:', DATA_DIR)

csv_files = sorted(glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True))
print(f'Found {len(csv_files)} CSV file(s)')
for f in csv_files:
    print(' -', os.path.basename(f))

In [ ]:
frames = []
for f in csv_files:
    try:
        frames.append(pd.read_csv(f, low_memory=False, encoding='latin-1'))
    except Exception as e:
        print(f'Skipped {f}: {e}')

df = pd.concat(frames, ignore_index=True)
df.columns = df.columns.str.strip()
print(f'Loaded {len(df):,} rows, {len(df.columns)} columns')
print('\nLabel column distribution:')
print(df['Label'].value_counts())

## 2. Clean & balance

In [ ]:
# Drop rows with inf/nan in feature columns or negative durations.
needed_cols = [
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std',
    'Packet Length Mean', 'Packet Length Std', 'Max Packet Length',
    'Flow Bytes/s', 'Total Fwd Packets', 'Total Backward Packets',
    'Label',
]
missing = [c for c in needed_cols if c not in df.columns]
if missing:
    print('WARNING — missing columns (will use defaults):', missing)

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[c for c in needed_cols if c in df.columns])
df = df[df['Flow Duration'] >= 0].reset_index(drop=True)
print(f'After cleaning: {len(df):,} rows')

In [ ]:
# Map labels → 3 actions (matches DQNAgent's action_dim=3 head).
ACTION_ALLOW = 0
ACTION_BLOCK = 1
ACTION_RATE_LIMIT = 2

RATE_LIMIT_LABELS = {'DoS slowloris', 'DoS Slowhttptest', 'Heartbleed'}

def label_to_action(label: str) -> int:
    label = str(label).strip()
    if label == 'BENIGN':
        return ACTION_ALLOW
    if label in RATE_LIMIT_LABELS:
        return ACTION_RATE_LIMIT
    return ACTION_BLOCK

df['action'] = df['Label'].apply(label_to_action)
print('Per-action sample counts:')
print(df['action'].value_counts().rename({0: 'ALLOW', 1: 'BLOCK', 2: 'RATE_LIMIT'}))

In [ ]:
# Balance classes — CIC-IDS2017 is ~80% benign. Downsample to roughly equal
# parts ALLOW vs (BLOCK + RATE_LIMIT) so the model doesn't drift back to
# block-everything-or-allow-everything.
TARGET_PER_CLASS = 80_000  # tune up/down if you want longer/shorter training

balanced = []
for a in (ACTION_ALLOW, ACTION_BLOCK, ACTION_RATE_LIMIT):
    subset = df[df['action'] == a]
    if len(subset) == 0:
        continue
    n = min(len(subset), TARGET_PER_CLASS)
    balanced.append(subset.sample(n=n, random_state=42, replace=False))

df_bal = pd.concat(balanced).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f'Balanced dataset: {len(df_bal):,} rows')
print(df_bal['action'].value_counts().rename({0: 'ALLOW', 1: 'BLOCK', 2: 'RATE_LIMIT'}))

## 3. Build the 12-dim state vector

Same shape and same `max_values` normalization the agent's `FlowManager._compile_state_vector` produces at runtime (see `agent/extraction/flow_manager.py:106-127`).

CIC-IDS2017 doesn't carry payload-entropy or IAT-autocorrelation columns, so those two dims get neutral defaults (0.5 / 0.0) — matching the NaN fallbacks in `flow_manager.py`.

In [ ]:
def col(df, name, default=0.0):
    return df[name].astype(float).values if name in df.columns else np.full(len(df), default)

def build_state_vector(df: pd.DataFrame) -> np.ndarray:
    n = len(df)

    # CIC IATs are in microseconds; agent expects seconds. Same for Flow Duration.
    iat_mean   = col(df, 'Flow IAT Mean') / 1e6
    iat_std    = col(df, 'Flow IAT Std')  / 1e6
    autocorr   = np.full(n, 0.5)                          # not in CIC — neutral
    size_mean  = col(df, 'Packet Length Mean')
    size_std   = col(df, 'Packet Length Std')
    size_max   = col(df, 'Max Packet Length')
    size_iqr   = col(df, 'Packet Length Std') * 1.349    # rough IQR proxy: ~1.35 * std for normal
    entropy_m  = np.full(n, 0.5)                          # not in CIC — neutral
    entropy_v  = np.full(n, 0.0)
    duration   = col(df, 'Flow Duration') / 1e6           # → seconds
    throughput = np.nan_to_num(col(df, 'Flow Bytes/s'), nan=0.0, posinf=0.0, neginf=0.0)
    pkt_count  = col(df, 'Total Fwd Packets') + col(df, 'Total Backward Packets')

    raw = np.column_stack([
        iat_mean, iat_std, autocorr,
        size_mean, size_std, size_max, size_iqr,
        entropy_m, entropy_v,
        duration, throughput, pkt_count,
    ])

    # Same max_values as flow_manager.py — keeps train/inference distributions aligned.
    max_values = np.array([
        1.0, 1.0, 1.0,
        1500, 500, 1500, 1500,
        1.0, 1.0,
        60.0, 1_000_000, 100,
    ], dtype=np.float64)

    normalized = raw / max_values
    return np.clip(normalized, 0.0, 1.0).astype(np.float32)

X = build_state_vector(df_bal)
y = df_bal['action'].astype(np.int64).values
print('X shape:', X.shape, 'dtype:', X.dtype)
print('y shape:', y.shape, 'unique:', np.unique(y, return_counts=True))
print('\nFeature stats (should be in [0,1]):')
print(pd.DataFrame(X).describe().T[['min', 'mean', 'max']].round(3))

## 4. Model & loss

`FirewallDQN` is copied verbatim from `agent/dqn/model.py`. We train it as a 3-class classifier with weighted cross-entropy — the cleanest way to bake "false positives on benign cost 10x more than missing an attack" into the loss without the gradient saturation that broke our earlier Q-regression attempt.

The Q-target table below is kept for reference (it documents the runtime reward shape the agent's human-feedback path uses) but isn't used in this training run.

In [ ]:
class FirewallDQN(nn.Module):
    def __init__(self, input_dim=12, output_dim=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# Reward table: reward_for[correct_action][agent_action]
#
# Asymmetric calibration: blocking a benign flow is 10x more costly than
# missing an attack (-200 vs -20). This mirrors real firewall economics —
# false positives lose customers; false negatives get caught downstream.
# Without this asymmetry the model lands at ~25% benign FP rate.
REWARD = {
    ACTION_ALLOW:      {ACTION_ALLOW:   1.0, ACTION_BLOCK: -200.0, ACTION_RATE_LIMIT: -30.0},
    ACTION_BLOCK:      {ACTION_ALLOW: -20.0, ACTION_BLOCK:   10.0, ACTION_RATE_LIMIT:   2.0},
    ACTION_RATE_LIMIT: {ACTION_ALLOW: -20.0, ACTION_BLOCK:    5.0, ACTION_RATE_LIMIT:  10.0},
}

# Pre-build Q-targets for each row: a 3-vector where entry i = reward if agent took action i.
# Regressing onto this teaches the network the same Q-landscape RL would converge to,
# but in supervised time (single pass through data, no replay buffer needed).
Q_targets = np.zeros((len(y), 3), dtype=np.float32)
for ca in (ACTION_ALLOW, ACTION_BLOCK, ACTION_RATE_LIMIT):
    mask = (y == ca)
    for a in (ACTION_ALLOW, ACTION_BLOCK, ACTION_RATE_LIMIT):
        Q_targets[mask, a] = REWARD[ca][a]

print('Q-target sample (first 3 rows):')
for i in range(3):
    print(f'  label={y[i]}  Q=[ALLOW={Q_targets[i,0]:+.1f}, BLOCK={Q_targets[i,1]:+.1f}, RATE_LIMIT={Q_targets[i,2]:+.1f}]')

## 5. Train

Weighted cross-entropy on the action label. Class weights = `[10, 1, 1]` so the optimizer pays 10x attention to ALLOW samples — making false-positive blocks the costliest error class.

In [ ]:
# Train/val split
perm = np.random.permutation(len(X))
split = int(0.9 * len(X))
train_idx, val_idx = perm[:split], perm[split:]

X_train = torch.from_numpy(X[train_idx]).to(device)
y_train = torch.from_numpy(y[train_idx]).long().to(device)
X_val   = torch.from_numpy(X[val_idx]).to(device)
y_val   = torch.from_numpy(y[val_idx]).long().to(device)

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=512, shuffle=True,
)

policy_net = FirewallDQN(input_dim=12, output_dim=3).to(device)
optimizer = optim.Adam(policy_net.parameters(), lr=5e-4)

EPOCHS = 40
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# Plain (balanced) cross-entropy. We learn the actual feature → label
# boundary here; the precision/recall tradeoff is handled post-hoc by
# tuning a bias on the ALLOW logit, which is more controllable than
# trying to express the tradeoff via class weights during training.
criterion = nn.CrossEntropyLoss()

for epoch in range(1, EPOCHS + 1):
    policy_net.train()
    total_loss = 0.0
    seen = 0
    for bx, by in train_loader:
        logits = policy_net(bx)
        loss = criterion(logits, by)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * bx.size(0)
        seen += bx.size(0)
    scheduler.step()

    policy_net.eval()
    with torch.no_grad():
        val_pred = policy_net(X_val).argmax(dim=1)
        val_acc = (val_pred == y_val).float().mean().item()
        benign_mask_t = (y_val == ACTION_ALLOW)
        attack_mask_t = (y_val != ACTION_ALLOW)
        fp_rate = ((val_pred != ACTION_ALLOW) & benign_mask_t).float().sum().item() / max(benign_mask_t.sum().item(), 1)
        catch   = ((val_pred != ACTION_ALLOW) & attack_mask_t).float().sum().item() / max(attack_mask_t.sum().item(), 1)
    lr_now = scheduler.get_last_lr()[0]
    print(f'Epoch {epoch:2d}/{EPOCHS}  lr={lr_now:.1e}  '
          f'train_loss={total_loss/seen:.4f}  val_acc={val_acc*100:.2f}%  '
          f'benign_FP={fp_rate*100:.2f}%  attack_catch={catch*100:.2f}%')

## 6. Tune the operating point (post-hoc bias sweep)

The trained model lives somewhere on a precision/recall curve. Adding a bias `b` to the ALLOW logit slides us along that curve:

- **`b > 0`** → predict ALLOW more often → fewer false positives, fewer attacks caught
- **`b < 0`** → predict BLOCK/RATE_LIMIT more often → more false positives, more attacks caught

We sweep `b` against the val set, pick the value that maximizes attack catch subject to `FP ≤ MAX_FP_RATE`, then **bake that bias into `fc3.bias[ALLOW]`**. The saved weights then encode the chosen operating point — agent inference stays as plain argmax, no runtime changes needed.

Adjust `MAX_FP_RATE` to slide along the tradeoff: lower = more conservative firewall (fewer false blocks but more missed attacks), higher = more aggressive.

In [ ]:
import numpy as np

# Set how much benign false-positive rate you're willing to tolerate.
# 0.05 = 5% of benign flows may be blocked; reasonable starting point.
# Lower → more conservative (fewer blocks, more missed attacks).
# Higher → more aggressive (catches more, but blocks more legit traffic).
MAX_FP_RATE = 0.05

policy_net.eval()
with torch.no_grad():
    val_logits = policy_net(X_val).cpu().numpy()
y_val_np = y_val.cpu().numpy()
benign_mask = (y_val_np == ACTION_ALLOW)
attack_mask = (y_val_np != ACTION_ALLOW)
n_benign = int(benign_mask.sum())
n_attack = int(attack_mask.sum())

candidates = []
for bias in np.arange(-5.0, 5.01, 0.1):
    shifted = val_logits.copy()
    shifted[:, ACTION_ALLOW] += bias
    pred = shifted.argmax(axis=1)
    fp = ((pred != ACTION_ALLOW) & benign_mask).sum() / max(n_benign, 1)
    catch = ((pred != ACTION_ALLOW) & attack_mask).sum() / max(n_attack, 1)
    candidates.append({'bias': float(bias), 'fp': float(fp), 'catch': float(catch)})

print(f'{"bias":>6s}  {"benign_FP":>10s}  {"attack_catch":>13s}')
print('-' * 35)
# Print a coarse sweep so you can see the full tradeoff curve.
for c in candidates[::5]:
    flag = '  <-- under FP target' if c['fp'] <= MAX_FP_RATE else ''
    print(f'{c["bias"]:+6.2f}  {c["fp"]*100:9.2f}%  {c["catch"]*100:12.2f}%{flag}')

under_target = [c for c in candidates if c['fp'] <= MAX_FP_RATE]
if under_target:
    best = max(under_target, key=lambda c: c['catch'])
    print(f'\nSelected bias = {best["bias"]:+.2f}  (highest catch with FP <= {MAX_FP_RATE*100:.0f}%)')
else:
    # Model couldn't hit the FP target at any bias — fall back to lowest FP.
    best = min(candidates, key=lambda c: c['fp'])
    print(f'\nWARNING: no bias hits FP <= {MAX_FP_RATE*100:.0f}%. Picking lowest-FP point.')
    print(f'Selected bias = {best["bias"]:+.2f}  (lowest achievable FP: {best["fp"]*100:.2f}%)')

# Bake the bias into the model's output layer. Mathematically equivalent
# to applying it at inference, but keeps the runtime agent code unchanged.
with torch.no_grad():
    policy_net.fc3.bias[ACTION_ALLOW] += best['bias']

# Re-verify post-baking
with torch.no_grad():
    val_pred = policy_net(X_val).argmax(dim=1).cpu().numpy()
fp_final = ((val_pred != ACTION_ALLOW) & benign_mask).sum() / max(n_benign, 1)
catch_final = ((val_pred != ACTION_ALLOW) & attack_mask).sum() / max(n_attack, 1)
print(f'After baking the bias into fc3.bias[ALLOW]:')
print(f'  benign FP rate    = {fp_final*100:.2f}%')
print(f'  attack catch rate = {catch_final*100:.2f}%')

## 7. Final confusion matrix

Reflects the model with the tuned bias baked in — i.e., the exact behavior you'll see at runtime after dropping `firewall_weights.pth` into the agent.

In [ ]:
policy_net.eval()
with torch.no_grad():
    val_pred = policy_net(X_val).argmax(dim=1).cpu().numpy()
val_true = y_val.cpu().numpy()

names = {0: 'ALLOW', 1: 'BLOCK', 2: 'RATE_LIMIT'}
cm = pd.crosstab(
    pd.Series([names[v] for v in val_true], name='truth'),
    pd.Series([names[v] for v in val_pred], name='pred'),
    margins=True,
)
print('Validation confusion matrix:')
print(cm)

# False-positive rate on benign (most important metric — controls how much
# normal traffic the deployed agent will incorrectly block).
benign_mask = (val_true == 0)
fp = ((val_pred != 0) & benign_mask).sum()
fp_rate = fp / max(benign_mask.sum(), 1)
print(f'\nBenign false-positive rate: {fp_rate*100:.2f}%  '
      f'({fp}/{benign_mask.sum()} benign flows would be blocked/rate-limited)')

# Attack catch rate.
attack_mask = (val_true != 0)
caught = ((val_pred != 0) & attack_mask).sum()
catch = caught / max(attack_mask.sum(), 1)
print(f'Attack catch rate:           {catch*100:.2f}%  '
      f'({caught}/{attack_mask.sum()} attack flows would be blocked/rate-limited)')

## 8. Save weights

Saves a `state_dict` PyTorch file. The agent's `DQNAgent.__init__` looks for this exact filename in its working directory (`agent/dqn/agent.py:25-28`) and loads it on startup.

Download from Kaggle's **Output** panel and replace `rl_firewall/agent/firewall_weights.pth`.

In [ ]:
out_path = '/kaggle/working/firewall_weights.pth'
torch.save(policy_net.state_dict(), out_path)
print(f'Saved {out_path}  ({os.path.getsize(out_path):,} bytes)')

# Quick load-test — verifies the file is loadable by the same architecture the agent uses.
test_net = FirewallDQN(input_dim=12, output_dim=3)
test_net.load_state_dict(torch.load(out_path, map_location='cpu'))
print('Load test: OK')